# 疾患関連遺伝子のスコアリング

候補リストから毎回ランダムに一群を取り出し、アルファベットのラベルを振って出題し、
遺伝子ごとのスコアを繰り返し平均する。

| | |
|---|---|
| ① | 疾患名を入力 |
| ② | ローカルで動いているモデルから選択 |
| ③ | 遺伝子リストを読み込む |
| ④ | プロンプト |
| ⑤ | `ranker()` — 1 群を採点して {遺伝子: スコア} を返す |
| ⑥ | ランダムに選んで繰り返し、平均する |
| ⑦ | DataFrame で表示 |

---

### 先に断っておく制約

**1. 1 群あたりの遺伝子は最大 26 個。** アルファベットは 26 文字しかないので、
27 個すべてに 1 文字を割り当てることはできません。2 文字（AA, AB…）にすると複数トークンになり、
次の 1 トークンだけを見る採点方式が壊れます。既定は 26 個（A〜Z）。
逃げ道（「どれでもない」）を入れるとその 1 文字ぶん減って 25 個です。`N_PER_ROUND` で変更できます。

**2. Ollama の `top_logprobs` は上限 20。** 26 ラベルすべての対数確率を 1 回で受け取ることはできません
（実測: 21 以上を指定すると `top_logprobs must be between 0 and 20` エラー）。
返ってこなかったラベルは下限値で埋めます。確率が 20 位に入らないラベルは元々ほぼ無視できる
大きさなので実害は小さいものの、**埋めた個数は毎回記録して ⑥ で表示します**。
ラベルの割り当ては毎回シャッフルし、特定の遺伝子が固定的に不利にならないようにします。

**3. 思考モデルは、そのままだと 1 文字を答えません。** qwen3 系は最初のトークンとして
`<think>` を出すため、A〜Z の対数確率が 1 つも取れません（実測で 0/26）。
これは黙って全部が下限値になり、**一見動いているのに中身が空**という壊れ方をします。
④ の指示文・few-shot と ⑤ の `think: false` は、この 3 つが揃って初めて letter が返るための
必須の部品です。外さないでください。⑤ の動作確認セルで、実際に返ったトークンを毎回表示します。


## 0. 準備


In [ ]:
import os, json, random, math, string, statistics, urllib.request, urllib.error
from collections import defaultdict, Counter

try:
    import pandas as pd
    HAVE_PANDAS = True
except ImportError:
    HAVE_PANDAS = False
    print("pandas がありません:  pip install pandas （表は簡易表示になります）")

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
print("Ollama:", OLLAMA_HOST)


## ① 疾患名


In [ ]:
DISEASE = "schizophrenia"      # ← ここに疾患名を書く

print("疾患:", DISEASE)


## ② モデルを選ぶ

ローカルの Ollama に入っているモデルを一覧します。`MODEL` に名前を入れてください。
空のままなら一覧の先頭を使います。


In [ ]:
def ollama_get(path):
    with urllib.request.urlopen(OLLAMA_HOST + path, timeout=10) as r:
        return json.loads(r.read().decode())


def list_local_models():
    """Ollama に入っているモデル名の一覧。埋め込み専用モデルは採点に使えないので外す。"""
    try:
        models = ollama_get("/api/tags").get("models", [])
    except urllib.error.URLError as e:
        print(f"Ollama に接続できません（{e.reason}）。`ollama serve` は動いていますか。")
        return []
    out = []
    for m in models:
        name = m["name"]
        if any(k in name.lower() for k in ("embed", "bge-", "e5-", "gte-")):
            continue          # 埋め込みモデルは logprobs を返さない
        out.append({"name": name,
                    "size_gb": round(m.get("size", 0) / 1e9, 1),
                    "params": m.get("details", {}).get("parameter_size", "?")})
    return out


AVAILABLE = list_local_models()
for i, m in enumerate(AVAILABLE):
    print(f"  [{i}] {m["name"]:<30}{m["params"]:>8}  {m["size_gb"]}GB")
if not AVAILABLE:
    print("  （使えるモデルが見つかりません）")


In [ ]:
MODEL = ""        # ← 例: "qwen3:14b"。空なら一覧の先頭

if not MODEL and AVAILABLE:
    MODEL = AVAILABLE[0]["name"]
names = [m["name"] for m in AVAILABLE]
if MODEL and names and MODEL not in names:
    print(f"⚠ {MODEL} は一覧にありません。`ollama pull {MODEL}` が要るかもしれません。")
print("モデル:", MODEL or "★未選択")


## ③ 遺伝子リストを読み込む


In [ ]:
GENE_FILE = "genelist01_500.txt"


def load_genes(path):
    """1 行 1 記号。# で始まる行と空行は読み飛ばす。重複は順序を保って落とす。"""
    seen, out, dupes = set(), [], 0
    with open(path) as f:
        for line in f:
            g = line.strip()
            if not g or g.startswith("#"):
                continue
            if g in seen:
                dupes += 1
                continue
            seen.add(g)
            out.append(g)
    return out, dupes


GENES, n_dupes = load_genes(GENE_FILE)
print(f"{GENE_FILE}: {len(GENES)} 遺伝子" + (f"（重複 {n_dupes} 件を除外）" if n_dupes else ""))
print("先頭:", GENES[:8])


## ④ プロンプト

モデルには **1 文字のラベルだけ**を答えさせ、遺伝子記号そのものは生成させません。
記号を書かせると GPR52 と GPR56 のような似た記号を取り違えるので、
選択肢は記号で提示し、対応付けはコード側で持ちます。

指示文と few-shot は飾りではありません。26 択という長い問いを前にすると、
モデルは 1 文字ではなく散文（`Answer`, `For`, `It`…）を書き始めます。
実測では、指示文だけ・few-shot だけでは足りず、両方に ⑤ の `think: false` を加えて
初めて letter が返りました。


In [ ]:
LETTERS = list(string.ascii_uppercase)      # A..Z

USE_EXIT     = False   # True にすると最後の 1 文字が「どれでもない」になる
N_PER_ROUND  = 26      # 1 群あたりの遺伝子数。上限は 26（USE_EXIT なら 25）

MAX_GENES = len(LETTERS) - (1 if USE_EXIT else 0)
if N_PER_ROUND > MAX_GENES:
    print(f"⚠ N_PER_ROUND={N_PER_ROUND} は上限 {MAX_GENES} を超えるので {MAX_GENES} に切り下げます。")
    N_PER_ROUND = MAX_GENES

INSTRUCTION = "Answer with a single letter only.\n\n"

# few-shot の正解は B と C。特定の文字に寄せないよう散らしてある。
FEWSHOT = (
    "Disease: Cystic fibrosis\n"
    "Options:\nA. HBB\nB. CFTR\nC. GPR52\nD. APOE\n"
    "Answer: B\n\n"
    "Disease: Sickle cell disease\n"
    "Options:\nA. CFTR\nB. APOE\nC. HBB\nD. GPR56\n"
    "Answer: C\n\n"
)


def build_prompt(disease, genes, use_exit=USE_EXIT, fewshot=True):
    """1 群 → プロンプト文字列と {ラベル: 遺伝子} の対応表。

    対応表を返すのが肝心なところ。ラベルは呼ぶたびに違う遺伝子を指すので、
    「A が正解」ではなく「A が指していたもの」を必ずこの表から読み戻すこと。"""
    head = INSTRUCTION + (FEWSHOT if fewshot else "")
    lines = [f"Disease: {disease}", "Options:"]
    mapping = {}
    for lab, g in zip(LETTERS, genes):
        lines.append(f"{lab}. {g}")
        mapping[lab] = g
    if use_exit:
        exit_lab = LETTERS[len(genes)]
        lines.append(f"{exit_lab}. None of the above")
        mapping[exit_lab] = None
    lines.append("Answer:")
    return head + "\n".join(lines), mapping


demo_prompt, demo_map = build_prompt(DISEASE, GENES[:N_PER_ROUND])
print(demo_prompt)


## ⑤ `ranker()` — 1 群を採点する

モデルに次の 1 トークンだけ生成させ、その位置の候補と対数確率を受け取ります。
ラベル A〜Z の対数確率を softmax で正規化し、遺伝子に付け替えて返します。

返ってこなかったラベルは、観測できた最小値からさらに下げた値で埋めます。
**埋めた個数と、実際に生成されたトークンも一緒に返します。**
生成トークンがラベルでない（`<think>` など）なら、そのラウンドは中身が空です。


In [ ]:
def _post(path, payload, timeout=180):
    req = urllib.request.Request(OLLAMA_HOST + path,
                                 data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"},
                                 method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())


def first_token_logprobs(prompt, model, top_logprobs=20, no_think=True):
    """次の 1 トークンの (生成トークン, {トークン: 対数確率})。

    top_logprobs は Ollama の上限が 20。no_think は思考モデル対策で、
    対応していないモデルに送ると弾かれることがあるのでその場合は外して再送する。"""
    payload = {"model": model, "prompt": prompt, "stream": False,
               "options": {"temperature": 0, "num_predict": 1},
               "logprobs": True, "top_logprobs": min(top_logprobs, 20)}
    if no_think:
        payload["think"] = False
    try:
        data = _post("/api/generate", payload)
    except urllib.error.HTTPError as e:
        if no_think:
            payload.pop("think", None)          # think 非対応モデル
            data = _post("/api/generate", payload)
        else:
            raise
    if "error" in data:
        raise RuntimeError(str(data["error"])[:200])
    lp = data.get("logprobs")
    if not lp:
        raise RuntimeError("logprobs が返りません。Ollama v0.12.11 以降が必要です。")
    first = lp[0]
    out = {}
    tok, val = first.get("token"), first.get("logprob")
    if tok is not None and val is not None:
        out[tok.strip()] = val
    for alt in (first.get("top_logprobs") or []):
        if isinstance(alt, dict):
            t, v = alt.get("token"), alt.get("logprob")
            if t is None:                      # {token: logprob} 形式のことがある
                for k, vv in alt.items():
                    out.setdefault(str(k).strip(), vv)
            elif v is not None:
                out.setdefault(str(t).strip(), v)
    return tok, out


def ranker(disease, genes, model, use_exit=USE_EXIT, shuffle_labels=True):
    """遺伝子の一群 → {遺伝子: スコア}。

    スコアはラベル集合の中で正規化した確率（0〜1、合計 1）。
    ラベルの割り当ては既定でシャッフルする。順番を固定すると、
    top-20 に入り損ねる位置の遺伝子が毎回同じになって偏るため。"""
    genes = list(genes)
    if shuffle_labels:
        random.shuffle(genes)
    prompt, mapping = build_prompt(disease, genes, use_exit)
    gen_token, raw = first_token_logprobs(prompt, model)

    labels = list(mapping)
    got = {lab: raw[lab] for lab in labels if lab in raw}
    base = {"n_labels": len(labels), "generated": gen_token,
            "n_missing": len(labels) - len(got)}
    if not got:
        # ラベルが 1 つも返っていない。全部を下限で埋めれば「動いた」ように
        # 見えてしまうので、ここでは空を返して呼び出し側に失敗と分からせる。
        return {**base, "scores": {}, "top": None, "top_prob": None}

    floor = min(got.values()) - 10.0        # 見えなかったラベルは下限に置く
    filled = {lab: got.get(lab, floor) for lab in labels}

    m = max(filled.values())
    exp = {lab: math.exp(v - m) for lab, v in filled.items()}
    z = sum(exp.values())
    prob = {lab: v / z for lab, v in exp.items()}

    scores = {mapping[lab]: p for lab, p in prob.items() if mapping[lab] is not None}
    top_lab = max(prob, key=prob.get)
    return {**base, "scores": scores,
            "top": mapping[top_lab],          # None なら「どれでもない」が 1 位
            "top_prob": prob[top_lab]}


In [ ]:
# 動作確認（1 群だけ）— ここで letter が返らなければ先へ進んでも無意味
if MODEL:
    random.seed(0)
    probe = ranker(DISEASE, GENES[:N_PER_ROUND], MODEL)
    print(f"生成トークン : {probe['generated']!r}")
    print(f"ラベル取得   : {probe['n_labels'] - probe['n_missing']}/{probe['n_labels']}"
          f"（{probe['n_missing']} 個は下限で補完。Ollama の上限 20 のため 6 個前後は正常）")
    if not probe["scores"]:
        print("\n❌ ラベルが 1 つも返っていません。以下を確認してください:")
        print("   ・思考モデルなら think:false が効いているか（生成トークンが <think> なら効いていない）")
        print("   ・④ の INSTRUCTION と FEWSHOT を消していないか")
        print("   ・別のモデルを試す")
    else:
        print(f"1 位         : {probe['top']}  (p={probe['top_prob']:.3f})")
        top5 = sorted(probe["scores"].items(), key=lambda kv: -kv[1])[:5]
        print("上位 5       :", [(g, round(p, 4)) for g, p in top5])
else:
    print("モデル未選択のため実行しません。")


## ⑥ 群を作って繰り返す

毎回 `N_PER_ROUND` 個を選んで `ranker()` に投げ、遺伝子ごとにスコアを溜めます。

### 全ての遺伝子を同じ回数だけ競争に入れる

毎回独立に無作為抽出すると、登場回数は二項分布になって大きくばらつきます。
501 遺伝子・1 群 26 個・100 ラウンドの実測で、**ランダム抽出では登場回数が 0〜13 回**
（平均 5.2、SD 2.3、一度も出ない遺伝子あり）。13 回出た遺伝子と 1 回しか出ない遺伝子の
平均スコアを同じ表で並べても、比べているのは実力ではなく試行回数です。

`balanced` は **山札方式**でこれを揃えます。全遺伝子を 1 つの山に切って上から 26 枚ずつ配り、
山が尽きたら切り直す。カードゲームで全員に同じ枚数を配るのと同じ理屈です。
同じ条件での実測は **5〜6 回（SD 0.39、未出現ゼロ）**、登場回数の差は常に最大 1 に収まります。

| | 登場回数の幅 | SD | 未出現 |
|---|---|---|---|
| ランダム | 0〜13 回 | 2.27 | 1 遺伝子 |
| 均等（山札） | 5〜6 回 | 0.39 | なし |

同じ群に同じ遺伝子が二度入らないよう、配る際に重複は飛ばします。
山を切り直すたびに相手の組み合わせは変わるので、**誰と当たるか**は依然ランダムです。
揃うのは回数だけで、対戦相手の強さまでは揃いません（そこは繰り返し回数で均します）。


In [ ]:
def random_groups(genes, n, rounds, rng):
    """毎回独立に無作為抽出。登場回数は揃わない。比較用。"""
    return [rng.sample(genes, min(n, len(genes))) for _ in range(rounds)]


def balanced_groups(genes, n, rounds, rng):
    """山札方式。全遺伝子を切って上から配り、尽きたら切り直す。

    どの遺伝子も、切り直しごとにちょうど 1 回ずつ配られる。したがって
    登場回数の差はラウンド数によらず最大 1。同一群内の重複だけは飛ばす。
    """
    n = min(n, len(genes))
    deck, out = [], []
    for _ in range(rounds):
        group = []
        while len(group) < n:
            if not deck:
                deck = list(genes)
                rng.shuffle(deck)
            picked = None
            for i, g in enumerate(deck):
                if g not in group:          # 同じ群に二度入れない
                    picked = deck.pop(i)
                    break
            if picked is None:              # 山の残りが全部この群にある
                deck = []
                continue
            group.append(picked)
        out.append(group)
    return out


def appearance_report(groups, genes, label):
    """登場回数が揃っているかを数字で見せる。揃っていないなら順位を読む前に直す。"""
    c = Counter(g for grp in groups for g in grp)
    counts = [c.get(g, 0) for g in genes]
    print(f"  {label:<16} min={min(counts):<3} max={max(counts):<3} "
          f"平均={statistics.mean(counts):.2f}  SD={statistics.pstdev(counts):.2f}  "
          f"未出現={sum(1 for x in counts if x == 0)}")
    return counts


In [ ]:
SAMPLING = "balanced"   # "balanced" = 山札方式（既定） / "random" = 毎回無作為
N_ROUNDS = 100          # ← 繰り返し回数。モデル呼び出し回数と同じ
SEED     = 0

ROUNDS_PER_EPOCH = -(-len(GENES) // N_PER_ROUND) if GENES else 0
print(f"{len(GENES)} 遺伝子 / 1 群 {N_PER_ROUND} 個")
print(f"全員を 1 周させるのに {ROUNDS_PER_EPOCH} ラウンド")

expected = N_ROUNDS * N_PER_ROUND / len(GENES) if GENES else 0
print(f"{N_ROUNDS} ラウンド → 1 遺伝子あたり平均 {expected:.1f} 回の登場")
if expected < 3:
    print("  ⚠ 少なすぎます。順位の差はほとんど試行回数の偶然です。")
if N_ROUNDS % ROUNDS_PER_EPOCH:
    nxt = (N_ROUNDS // ROUNDS_PER_EPOCH + 1) * ROUNDS_PER_EPOCH
    print(f"  ヒント: {ROUNDS_PER_EPOCH} の倍数（次は {nxt}）にすると全員が完全に同じ回数になります。")

# 実際にモデルへ投げる前に、群の作り方だけ確認しておく
if GENES:
    print("\n登場回数の分布（モデル呼び出しなし・シミュレーション）:")
    for name, fn in (("random", random_groups), ("balanced", balanced_groups)):
        appearance_report(fn(GENES, N_PER_ROUND, N_ROUNDS, random.Random(SEED)),
                          GENES, name)


In [ ]:
def run_rounds(disease, genes, model, n_rounds, n_per_round, seed=0,
               sampling="balanced", progress_every=10):
    """繰り返して {遺伝子: [スコア,...]} を集める。

    群の作り方を先に全部決めてから回す。こうしておくと、途中で失敗した回が
    あっても「どの遺伝子が何回出題されたはずか」が後から分かる。"""
    rng = random.Random(seed)
    make = balanced_groups if sampling == "balanced" else random_groups
    groups = make(genes, n_per_round, n_rounds, rng)

    collected = defaultdict(list)
    wins = defaultdict(int)
    missing, none_wins, failures, empty = [], 0, 0, 0

    for i, group in enumerate(groups):
        try:
            res = ranker(disease, group, model)
        except Exception as e:
            failures += 1
            print(f"  round {i}: 失敗 {type(e).__name__}: {e}")
            continue
        if not res["scores"]:
            empty += 1
            continue
        for g, p in res["scores"].items():
            collected[g].append(p)
        if res["top"] is None:
            none_wins += 1
        else:
            wins[res["top"]] += 1
        missing.append(res["n_missing"])
        if progress_every and (i + 1) % progress_every == 0:
            print(f"  {i + 1}/{n_rounds} 回", flush=True)

    return {"collected": collected, "wins": wins, "missing": missing,
            "none_wins": none_wins, "failures": failures, "empty": empty,
            "groups": groups, "sampling": sampling}


run = None
if MODEL and GENES:
    run = run_rounds(DISEASE, GENES, MODEL, N_ROUNDS, N_PER_ROUND, SEED,
                     sampling=SAMPLING)
    print(f"\n完了（{run['sampling']}）。通信失敗 {run['failures']} 回 / "
          f"ラベルが取れなかった回 {run['empty']} 回")
    if run["missing"]:
        print(f"補完したラベル数: 中央値 {statistics.median(run['missing']):.0f} / "
              f"{N_PER_ROUND + (1 if USE_EXIT else 0)}")
    if USE_EXIT:
        print(f"「どれでもない」が 1 位だった回: {run['none_wins']}/{N_ROUNDS}")

    # 出題された回数と、実際に採点できた回数は失敗のぶんだけずれる
    asked = Counter(g for grp in run["groups"] for g in grp)
    scored = {g: len(v) for g, v in run["collected"].items()}
    print(f"出題された遺伝子: {len(asked)}/{len(GENES)}  "
          f"（出題回数 {min(asked.values())}〜{max(asked.values())} 回）")
    print(f"採点された遺伝子: {len(scored)}/{len(GENES)}")
else:
    print("モデル未選択、または遺伝子リストが空です。")


## ⑦ 遺伝子ごとのスコア


In [ ]:
def to_rows(run, genes):
    """平均スコアの降順。ばらつきと登場回数を必ず併記する。

    平均だけを見ると、2 回しか出ていない遺伝子と 30 回出ている遺伝子が
    同じ確からしさに見えてしまう。"""
    rows = []
    for g in genes:
        vals = run["collected"].get(g, [])
        if not vals:
            rows.append({"gene": g, "mean_score": None, "sd": None,
                         "n_seen": 0, "n_wins": 0, "max_score": None})
            continue
        rows.append({
            "gene": g,
            "mean_score": statistics.mean(vals),
            "sd": statistics.stdev(vals) if len(vals) > 1 else 0.0,
            "n_seen": len(vals),
            "n_wins": run["wins"].get(g, 0),
            "max_score": max(vals),
        })
    rows.sort(key=lambda r: (r["mean_score"] is None, -(r["mean_score"] or 0)))
    for i, r in enumerate(rows, 1):
        r["rank"] = i
    return rows


rows = to_rows(run, GENES) if run else []
df = None

if rows and HAVE_PANDAS:
    df = pd.DataFrame(rows)[
        ["rank", "gene", "mean_score", "sd", "n_seen", "n_wins", "max_score"]]
    df = df.round({"mean_score": 5, "sd": 5, "max_score": 5})
    try:
        display(df.head(30))
    except NameError:
        print(df.head(30).to_string(index=False))
elif rows:
    for r in rows[:30]:
        ms = "-" if r["mean_score"] is None else f"{r['mean_score']:.5f}"
        print(f"{r['rank']:>4}  {r['gene']:<12}{ms:>10}  "
              f"n={r['n_seen']:<4}wins={r['n_wins']}")
else:
    print("結果がありません。")


### 保存


In [ ]:
import csv as _csv

if rows:
    out = f"{DISEASE.replace(' ', '_')}_scores.csv"
    if df is not None:
        df.to_csv(out, index=False)
    else:
        with open(out, "w", newline="") as fh:
            cols = ["rank", "gene", "mean_score", "sd", "n_seen", "n_wins", "max_score"]
            w = _csv.DictWriter(fh, fieldnames=cols, extrasaction="ignore")
            w.writeheader(); w.writerows(rows)
    print("書き出しました:", out)


## 数字を信じる前に

- **`n_seen` が小さい行は読まないでください。** 登場回数が数回の遺伝子の平均は、
  どの 25 個と同じ群に入ったかでいくらでも動きます。上位を語るなら、
  少なくとも全遺伝子が 10 回以上登場する回数までまわしてください。
- **これは相対評価です。** スコアは群の中で正規化した確率なので、
  弱い候補ばかりの群に入れば弱い遺伝子でも 1 位になります。絶対的な確信度ではありません。
- **対照を取らないと意味は分かりません。** 疾患名を無関係なものに差し替えて同じ順位が出るなら、
  疾患ではなく知名度を読んでいます。文献の多い遺伝子（TP53, EGFR, TNF）が
  どの疾患でも上位に来ていないかは必ず確認してください。
- **`genelist01_500.txt` は HGNC からの無作為抽出で、特定の疾患用に選んでいません。**
  記号はすべて実在しますが、既知の正解が入っている保証はありません。上位に出た遺伝子は
  「この 500 個の中では相対的に高い」という以上の意味を持ちません。
  実際に使うときは候補リストを差し替えてください。
- **⑤ の動作確認を毎回見てください。** 生成トークンがラベルでないとき、この方式は
  エラーを出さずに沈黙して壊れます。モデルを変えたら必ず確認し直してください。
